In [ ]:

!pip install -q transformers accelerate torch

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import numpy as np


# PATHS (adjust if different)

CLS_MODEL_PATH = "/kaggle/input/classification-trained-model/pytorch/default/1"
REG_MODEL_PATH = "/kaggle/input/regression-trained-model/pytorch/default/1"


# LOAD MODELS & TOKENIZER

tokenizer = AutoTokenizer.from_pretrained(CLS_MODEL_PATH)

# Load classification model
cls_model = AutoModelForSequenceClassification.from_pretrained(CLS_MODEL_PATH)
cls_model.eval()

# Load regression model
reg_model = AutoModelForSequenceClassification.from_pretrained(REG_MODEL_PATH)
reg_model.config.problem_type = "regression"
reg_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cls_model.to(device)
reg_model.to(device)


# Define severity labels (ensure same order as during training)

severity_labels = ["Low", "Medium", "High", "Critical"]  #  adjust if different


# Inference Function

def predict_vulnerability(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)

    # ---- Classification Prediction ----
    with torch.no_grad():
        cls_outputs = cls_model(**inputs)
        probs = F.softmax(cls_outputs.logits, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()
        severity = severity_labels[pred_class]
        confidence = probs[0][pred_class].item()

    # ---- Regression Prediction ----
    with torch.no_grad():
        reg_outputs = reg_model(**inputs)
        cvss_score = reg_outputs.logits.squeeze().item()
        cvss_score = np.clip(cvss_score, 0.0, 10.0)

    return severity, confidence, cvss_score


# User Input and Prediction

print("=== CVSS Severity & Score Prediction ===\n")
user_input = input("Enter vulnerability description:\n> ")

if user_input.strip():
    severity, confidence, score = predict_vulnerability(user_input)
    print("\n=====  PREDICTION RESULT =====")
    print(f" Severity: {severity} (confidence: {confidence:.2%})")
    print(f" CVSS Score: {score:.2f}")
else:
    print(" No input entered. Pleūase provide a valid vulnerability description.")


In [ ]:
import json
import os

# Replace this with your extracted model folder path
model_path = "/kaggle/input/regression-trained-model/pytorch/default/1"

config_path = os.path.join(model_path, "config.json")

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = json.load(f)

    print(" Architecture:", config.get("architectures"))
    print(" Problem Type:", config.get("problem_type"))
    print(" Number of Labels:", config.get("num_labels"))

    if config.get("problem_type") == "regression" or config.get("num_labels") == 1:
        print(" This is a Regression Model (predicts CVSS score).")
    else:
        print(" This is a Classification Model (predicts Severity class).")
else:
    print(" config.json not found! Make sure you extracted the model folder.")


In [ ]:

# Inference Script for Severity Classification + CVSS Regression


import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np


# Paths (update if needed)

CLASS_MODEL_PATH = "/kaggle/input/classification-trained-model/pytorch/default/1"
REG_MODEL_PATH   = "/kaggle/input/regression-trained-model/pytorch/default/1"

# Load both models
print(" Loading models...")
tokenizer_cls = AutoTokenizer.from_pretrained(CLASS_MODEL_PATH)
model_cls = AutoModelForSequenceClassification.from_pretrained(CLASS_MODEL_PATH)

tokenizer_reg = AutoTokenizer.from_pretrained(REG_MODEL_PATH)
model_reg = AutoModelForSequenceClassification.from_pretrained(REG_MODEL_PATH)

model_cls.eval()
model_reg.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_cls.to(device)
model_reg.to(device)
print(f" Models loaded on {device.upper()}")


# Label mapping (must match training)

label_names = ["None", "Low", "Medium", "High", "Critical"]  # adjust if needed


# Preprocessing function

def preprocess(text, tokenizer, max_len=256):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_len)
    return {k: v.to(device) for k, v in inputs.items()}


# Prediction function

def predict_cve(text):
    # ---- Classification ----
    inputs_cls = preprocess(text, tokenizer_cls)
    with torch.no_grad():
        outputs_cls = model_cls(**inputs_cls)
        preds_cls = torch.argmax(outputs_cls.logits, dim=1).cpu().item()
        severity = label_names[preds_cls]

    # ---- Regression ----
    inputs_reg = preprocess(text, tokenizer_reg)
    with torch.no_grad():
        outputs_reg = model_reg(**inputs_reg)
        score = outputs_reg.logits.squeeze().cpu().item()
        score = float(np.clip(score, 0.0, 10.0))  # ensure 0–10 range

    return severity, score


# Interactive Input

while True:
    user_input = input("\nEnter vulnerability description (or 'exit' to quit):\n> ")
    if user_input.lower() in ["exit", "quit"]:
        break

    severity, score = predict_cve(user_input)
    print("\n Prediction Results:")
    print(f"  Severity Level: {severity}")
    print(f"  CVSS Base Score: {score:.2f}")


 Loading models...


2025-11-14 07:09:51.845875: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763104191.868823     138 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763104191.875591     138 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


 Models loaded on CUDA



Enter vulnerability description (or 'exit' to quit):
>   An input validation bypass allows attackers to access restricted files using directory traversal sequences.



 Prediction Results:
  Severity Level: High
  CVSS Base Score: 5.52
